## Импорты библиотек

In [1]:
import math
import random
import warnings
import joblib
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import ast
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import shap

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

## Конфигурация путей и фиксация seed

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATA_DIR = Path('data')
ARTIFACTS_DIR = Path('artifacts')

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## Чтение датасета сотрудников

**Входные данные о сотрудниках будут подаваться через API**

In [3]:
employees_path = DATA_DIR / 'employees_train.csv'

if not employees_path.exists():
    raise FileNotFoundError(f"Файл {employees_path} не найден.")

df_employees = pd.read_csv(employees_path, encoding='utf-8-sig')

list_columns = ['languages', 'technologies']
for col in list_columns:
    if col in df_employees.columns:
        df_employees[col] = df_employees[col].apply(
            lambda x: ast.literal_eval(x) if pd.notna(x) and isinstance(x, str) else x
        )

date_columns = ['hire_date', 'last_promotion_date']
for col in date_columns:
    if col in df_employees.columns:
        df_employees[col] = pd.to_datetime(df_employees[col], errors='coerce')

print(f"Прочитано {len(df_employees)} данных о сотрудниках")

Прочитано 200 данных о сотрудниках


## Генерация обучающей выборки

Формируется датасет из проектных конфигураций. Целевая переменная рассчитывается по формуле, описанной в концепции

In [4]:
FEATURE_COLS = [
    'team_size', 'skill_coverage', 'avg_performance', 'avg_engagement', 'avg_turnover_risk', 'senior_ratio', 'budget_fit', 'timeline_days', 'budget_limit'
]

def generate_training_data(df_emp: pd.DataFrame, n_projects: int, seed: int = 42) -> pd.DataFrame:
    np.random.seed(seed)
    random.seed(seed)
    
    PROJECT_PROFILES = [
        {'timeline_days': 90,  'budget_limit': 4_500_000, 'req_skills': {'Python', 'SQL', 'Go', 'DevOps', 'Cloud', 'ML', 'Data Engineering'}},
        {'timeline_days': 60,  'budget_limit': 3_000_000, 'req_skills': {'Python', 'SQL', 'JavaScript', 'Analytics', 'Automation'}},
        {'timeline_days': 120, 'budget_limit': 6_000_000, 'req_skills': {'Go', 'Java', 'Cloud', 'DevOps', 'Kubernetes'}},
        {'timeline_days': 45,  'budget_limit': 2_500_000, 'req_skills': {'R', 'Python', 'SQL', 'ML', 'BI Dashboards'}},
        {'timeline_days': 100, 'budget_limit': 5_000_000, 'req_skills': {'Java', 'SQL', 'C#', 'Security', 'Data Engineering'}}
    ]
    
    records = []
    emp_ids = df_emp['employee_id'].tolist()
    
    for _ in range(n_projects):
        profile = random.choice(PROJECT_PROFILES)
        team_size = random.randint(3, 5)
        team = df_emp[df_emp['employee_id'].isin(random.sample(emp_ids, k=team_size))]
        
        lang_set = set().union(*[set(l) for l in team['languages'] if isinstance(l, (list, tuple))])
        tech_set = set().union(*[set(t) for t in team['technologies'] if isinstance(t, (list, tuple))])
        team_skills = lang_set.union(tech_set)
        
        skill_coverage = len(profile['req_skills'].intersection(team_skills)) / len(profile['req_skills'])
        estimated_cost = team['base_salary_monthly'].sum() * (profile['timeline_days'] / 30.0)
        budget_fit = 1.0 if estimated_cost <= profile['budget_limit'] else 0.65
        
        success_score = (
            0.30 * skill_coverage +
            0.25 * (team['performance_rating'].mean() / 5.0) +
            0.15 * (team['engagement_score'].mean() / 5.0) +
            0.15 * (1.0 - team['turnover_risk_score'].mean()) +
            0.10 * (team['job_level'].isin(['L4', 'L5', 'L6'])).mean() +
            0.05 * budget_fit +
            0.05 * (profile['timeline_days'] / 120.0) +
            np.random.normal(0, 0.04)
        )
        
        records.append({
            'team_size': team_size, 'skill_coverage': round(skill_coverage, 3),
            'avg_performance': round(team['performance_rating'].mean(), 3),
            'avg_engagement': round(team['engagement_score'].mean(), 3),
            'avg_turnover_risk': round(team['turnover_risk_score'].mean(), 3),
            'senior_ratio': round((team['job_level'].isin(['L4','L5','L6'])).mean(), 3),
            'budget_fit': budget_fit, 'timeline_days': profile['timeline_days'],
            'budget_limit': profile['budget_limit'], 'project_success': np.clip(success_score, 0.35, 0.95)
        })
    return pd.DataFrame(records)

Генерация 800 проектных конфигураций

In [5]:
num_training_data = 800

df_train = generate_training_data(df_employees, n_projects=num_training_data)
print(f"Сгенерировано {len(df_train)} конфигураций для обучения.")

Сгенерировано 800 конфигураций для обучения.


Вывод 5 конфигураций

In [6]:
df_train.head()

,team_size,skill_coverage,avg_performance,avg_engagement,avg_turnover_risk,senior_ratio,budget_fit,timeline_days,budget_limit,project_success
0,3,0.571,3.370,4.067,0.164,0.000,1.0,90,4500000,0.694647
1,3,0.600,3.080,3.550,0.164,0.000,1.0,60,3000000,0.635319
2,3,0.400,3.373,4.047,0.216,0.667,1.0,100,5000000,0.711858
3,3,0.286,2.933,3.800,0.193,0.000,1.0,90,4500000,0.615802
4,3,1.000,3.073,4.463,0.194,0.333,1.0,100,5000000,0.824051


## Обучение и тестирование модели

Используется RandomForestRegressor

Выборка разделяется в пропорции 80/20

Модель оценивается по метрикам MAE и R^2

In [7]:
def train_and_evaluate_model(
    df_train: pd.DataFrame, 
    test_size: float = 0.2, 
    random_state: int = 42
) -> Tuple[RandomForestRegressor, Dict[str, float]]:
    
    X = df_train[FEATURE_COLS]
    y = df_train['project_success']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    model = RandomForestRegressor(n_estimators=150, max_depth=6, random_state=random_state, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    metrics = {'MAE': mean_absolute_error(y_test, y_pred), 'R2': r2_score(y_test, y_pred)}
    print(f"Метрики на тестовой выборке: MAE={metrics['MAE']:.4f}, R^2={metrics['R2']:.4f}")
    
    return model, metrics

In [8]:
model, train_metrics = train_and_evaluate_model(df_train)

Метрики на тестовой выборке: MAE=0.0371, R^2=0.7124


In [9]:
MODEL_FILENAME = 'rf_model.joblib'
model_path = ARTIFACTS_DIR / MODEL_FILENAME

joblib.dump(model, model_path)

print(f"Модель сохранена в {model_path.resolve()}")

Модель сохранена в D:\projects\academic\dss-team-formation\artifacts\rf_model.joblib
